### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [3]:
from unsloth import FastModel
import torch

MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 2048  # matches the Gemma 3 Menochat run; bump later if needed

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    dtype = None,            # auto detection (bf16 on Ampere+)
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,     # QLoRA
    full_finetuning = False,
    # token = "YOUR_HF_TOKEN",  # if gated
)

print("Loaded model:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model dtype:", next(model.parameters()).dtype)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.5: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.367 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Loaded model: unsloth/gemma-4-E4B-it
Tokenizer class: Gemma4Processor
Model dtype: torch.bfloat16
CUDA available: True
GPU: NVIDIA GeForce RTX 5090


# Finetune Gemma 4 E4B on Menochat (text-only)

This notebook is a port of the Unsloth Gemma 4 E4B finetune template, adapted
to train the Menochat Bangla women's-health assistant on the same JSONL
dataset and same hyperparameters used for the Gemma 3 4B baseline. Vision and
audio towers are frozen; only the text decoder is LoRA-tuned.


We now add LoRA adapters so we only need to update a small amount of parameters!

In [4]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # text-only finetune for Menochat v1
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 16,              # matches Gemma 3 Menochat run
    lora_alpha = 32,     # matches Gemma 3 Menochat run
    lora_dropout = 0.05, # matches Gemma 3 Menochat run
    bias = "none",
    random_state = 3407,
)

print("LoRA attached successfully.")


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


LoRA attached successfully.


<a name="Data"></a>
### Data Prep
We now use the `Gemma-4` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-4 renders multi turn conversations like below:

```
<bos><|turn>user
Hello<turn|>
<|turn>model
Hey there!<turn|>
```
We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3, gemma-4` and more.

In [5]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

In [6]:
tokenizer

Gemma4Processor:
- feature_extractor: Gemma4AudioFeatureExtractor {
  "dither": 0.0,
  "feature_extractor_type": "Gemma4AudioFeatureExtractor",
  "feature_size": 128,
  "fft_length": 512,
  "fft_overdrive": false,
  "frame_length": 320,
  "hop_length": 160,
  "input_scale_factor": 1.0,
  "max_frequency": 8000.0,
  "mel_floor": 0.001,
  "min_frequency": 0.0,
  "padding_side": "left",
  "padding_value": 0.0,
  "per_bin_mean": null,
  "per_bin_stddev": null,
  "preemphasis": 0.0,
  "preemphasis_htk_flavor": true,
  "return_attention_mask": true,
  "sampling_rate": 16000
}

- image_processor: Gemma4ImageProcessor {
  "do_convert_rgb": true,
  "do_normalize": false,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.0,
    0.0,
    0.0
  ],
  "image_processor_type": "Gemma4ImageProcessor",
  "image_seq_length": 280,
  "image_std": [
    1.0,
    1.0,
    1.0
  ],
  "max_soft_tokens": 280,
  "patch_size": 16,
  "pooling_kernel_size": 3,
  "resample": 3,
  "rescale_factor": 0.

### Load the Menochat JSONL dataset

Replaces Unsloth's FineTome demo with the actual Menochat train/val JSONL files
used in the Gemma 3 run. Format per line: `{"messages": [{"role": "user"|"model",
"content": "..."}, ...]}`.


In [7]:
import json
from pathlib import Path
from datasets import Dataset

TRAIN_PATH = Path("/workspace/data_training/train.jsonl")
VAL_PATH   = Path("/workspace/data_training/val.jsonl")

print("TRAIN exists:", TRAIN_PATH.exists(), TRAIN_PATH)
print("VAL   exists:", VAL_PATH.exists(),   VAL_PATH)


def load_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                print(f"Bad JSON at line {i} in {path.name}: {e}")
                raise
    return rows


train_rows = load_jsonl(TRAIN_PATH)
val_rows   = load_jsonl(VAL_PATH)

print("Train rows:", len(train_rows))
print("Val rows  :", len(val_rows))
print("\nSample train keys:", train_rows[0].keys())
print("\nSample train row  :")
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2)[:2000])


def validate_messages(rows, split_name):
    bad = 0
    for idx, row in enumerate(rows):
        if "messages" not in row or not isinstance(row["messages"], list) or not row["messages"]:
            bad += 1
            if bad <= 3:
                print(f"[{split_name}] bad row {idx}")
            continue
        for m in row["messages"]:
            if not isinstance(m, dict) or "role" not in m or "content" not in m:
                bad += 1
                if bad <= 3:
                    print(f"[{split_name}] bad message at row {idx}: {m}")
                break
    print(f"[{split_name}] bad rows: {bad}")


validate_messages(train_rows, "train")
validate_messages(val_rows,   "val")

train_ds = Dataset.from_list(train_rows)
val_ds   = Dataset.from_list(val_rows)

print("\nDataset objects created.")
print(train_ds)
print(val_ds)


TRAIN exists: True /workspace/data_training/train.jsonl
VAL   exists: True /workspace/data_training/val.jsonl
Train rows: 5187
Val rows  : 300

Sample train keys: dict_keys(['id', 'messages'])

Sample train row  :
{
  "id": "q_00000",
  "messages": [
    {
      "role": "user",
      "content": "আমার ইদানীং মাসিকের সময় খুব অস্বস্তিকর ব্যথা হচ্ছে। আমি ভাবছিলাম, শ্বাস-প্রশ্বাসের ব্যায়াম কি এই ব্যথা কমাতে সাহায্য করতে পারে?"
    },
    {
      "role": "assistant",
      "content": "আপু, আপনার প্রশ্নটা খুবই প্রাসঙ্গিক। হ্যাঁ, শ্বাস-প্রশ্বাসের ব্যায়াম মাসিকের ব্যথা কমাতে সত্যিই সাহায্য করতে পারে। যখন আমাদের ব্যথা হয়, তখন শরীর আপনাআপনিই টানটান হয়ে যায় এবং পেশীগুলো সংকুচিত হয়, যা ব্যথাকে আরও বাড়িয়ে তোলে। গভীর এবং ধীর শ্বাস-প্রশ্বাসের ব্যায়াম করলে শরীর শিথিল হয়, পেশীগুলো আলগা হয় এবং মন শান্ত হয়। এর ফলে ব্যথার তীব্রতা কিছুটা কমে আসে এবং আপনি আরও স্বস্তি বোধ করতে পারেন। মাসিকের ব্যথা কমানোর জন্য আপনি কিছু সহজ উপায় অবলম্বন করতে পারেন: * **গভীর শ্বাস-প্রশ্বাস:** আরামদায়ক ভঙ্গিতে বসুন বা শুয়ে পড়ু

The Menochat JSONL is already in the standard HF chat format (`messages` with
`role`/`content`), so we skip `standardize_data_formats` and apply the chat
template directly in the next cell.


In [ ]:
# Menochat data is already in the correct {"messages": [...]} format,
# so standardize_data_formats is not needed. Left here commented for reference:
#
# from unsloth.chat_templates import standardize_data_formats
# train_ds = standardize_data_formats(train_ds)
# val_ds   = standardize_data_formats(val_ds)


Let's peek at a Menochat training row to confirm the structure before
applying the chat template.


In [8]:
train_ds[0]["messages"]


[{'content': 'আমার ইদানীং মাসিকের সময় খুব অস্বস্তিকর ব্যথা হচ্ছে। আমি ভাবছিলাম, শ্বাস-প্রশ্বাসের ব্যায়াম কি এই ব্যথা কমাতে সাহায্য করতে পারে?',
  'role': 'user'},
 {'content': 'আপু, আপনার প্রশ্নটা খুবই প্রাসঙ্গিক। হ্যাঁ, শ্বাস-প্রশ্বাসের ব্যায়াম মাসিকের ব্যথা কমাতে সত্যিই সাহায্য করতে পারে। যখন আমাদের ব্যথা হয়, তখন শরীর আপনাআপনিই টানটান হয়ে যায় এবং পেশীগুলো সংকুচিত হয়, যা ব্যথাকে আরও বাড়িয়ে তোলে। গভীর এবং ধীর শ্বাস-প্রশ্বাসের ব্যায়াম করলে শরীর শিথিল হয়, পেশীগুলো আলগা হয় এবং মন শান্ত হয়। এর ফলে ব্যথার তীব্রতা কিছুটা কমে আসে এবং আপনি আরও স্বস্তি বোধ করতে পারেন। মাসিকের ব্যথা কমানোর জন্য আপনি কিছু সহজ উপায় অবলম্বন করতে পারেন: * **গভীর শ্বাস-প্রশ্বাস:** আরামদায়ক ভঙ্গিতে বসুন বা শুয়ে পড়ুন। ধীরে ধীরে নাক দিয়ে গভীর শ্বাস নিন, পেট ফুলিয়ে তুলুন। কয়েক সেকেন্ড শ্বাস ধরে রেখে তারপর ধীরে ধীরে মুখ দিয়ে শ্বাস ছাড়ুন। এভাবে কয়েকবার করুন। এটি মনকে শান্ত করে এবং পেশী শিথিল করতে সাহায্য করে। * **গরম সেঁক:** পেটে বা পিঠে গরম জলের বোতল বা গরম প্যাড ব্যবহার করলে পেশী শিথিল হয় এবং ব্যথা কমে। * **হালকা ব্যায়

Apply the Gemma 4 chat template to each row's `messages` list. We keep the
`.removeprefix("<bos>")` guard because the rendered sample from this dataset
shows a leading `<bos>` — the processor tokenizer will re-add BOS at
tokenization time, and we do not want double BOS.


In [9]:
def formatting_prompts_func(examples):
    msgs_list = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        ).removeprefix("<bos>")
        for msgs in msgs_list
    ]
    return {"text": texts}


train_ds = train_ds.map(formatting_prompts_func, batched=True, desc="Formatting train with chat template")
val_ds   = val_ds.map(formatting_prompts_func,   batched=True, desc="Formatting val with chat template")

print(train_ds)
print(val_ds)


Formatting train with chat template:   0%|          | 0/5187 [00:00<?, ? examples/s]

Formatting val with chat template:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'messages', 'text'],
    num_rows: 5187
})
Dataset({
    features: ['id', 'messages', 'text'],
    num_rows: 300
})


Let's inspect one fully-formatted training example to make sure the chat
template rendered correctly and has no stray BOS at position 0.


In [10]:
print(train_ds[0]["text"][:3000])


<|turn>user
আমার ইদানীং মাসিকের সময় খুব অস্বস্তিকর ব্যথা হচ্ছে। আমি ভাবছিলাম, শ্বাস-প্রশ্বাসের ব্যায়াম কি এই ব্যথা কমাতে সাহায্য করতে পারে?<turn|>
<|turn>model
আপু, আপনার প্রশ্নটা খুবই প্রাসঙ্গিক। হ্যাঁ, শ্বাস-প্রশ্বাসের ব্যায়াম মাসিকের ব্যথা কমাতে সত্যিই সাহায্য করতে পারে। যখন আমাদের ব্যথা হয়, তখন শরীর আপনাআপনিই টানটান হয়ে যায় এবং পেশীগুলো সংকুচিত হয়, যা ব্যথাকে আরও বাড়িয়ে তোলে। গভীর এবং ধীর শ্বাস-প্রশ্বাসের ব্যায়াম করলে শরীর শিথিল হয়, পেশীগুলো আলগা হয় এবং মন শান্ত হয়। এর ফলে ব্যথার তীব্রতা কিছুটা কমে আসে এবং আপনি আরও স্বস্তি বোধ করতে পারেন। মাসিকের ব্যথা কমানোর জন্য আপনি কিছু সহজ উপায় অবলম্বন করতে পারেন: * **গভীর শ্বাস-প্রশ্বাস:** আরামদায়ক ভঙ্গিতে বসুন বা শুয়ে পড়ুন। ধীরে ধীরে নাক দিয়ে গভীর শ্বাস নিন, পেট ফুলিয়ে তুলুন। কয়েক সেকেন্ড শ্বাস ধরে রেখে তারপর ধীরে ধীরে মুখ দিয়ে শ্বাস ছাড়ুন। এভাবে কয়েকবার করুন। এটি মনকে শান্ত করে এবং পেশী শিথিল করতে সাহায্য করে। * **গরম সেঁক:** পেটে বা পিঠে গরম জলের বোতল বা গরম প্যাড ব্যবহার করলে পেশী শিথিল হয় এবং ব্যথা কমে। * **হালকা ব্যায়াম:** হালকা হাঁটা

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [11]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/workspace/gemma4_e4b_bn_menochat_unsloth_run1"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,

        output_dir = OUTPUT_DIR,
        num_train_epochs = 2,

        per_device_train_batch_size = 1,
        per_device_eval_batch_size  = 1,
        gradient_accumulation_steps = 8,  # effective batch 8

        learning_rate     = 1e-4,
        lr_scheduler_type = "cosine",
        warmup_ratio      = 0.05,

        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps    = 50,
        save_strategy = "steps",
        save_steps    = 50,
        save_total_limit = 2,

        bf16 = True,
        fp16 = False,

        optim = "adamw_8bit",
        weight_decay = 0.001,
        gradient_checkpointing = True,

        # NOTE: we intentionally do NOT set add_special_tokens=False here.
        # Flow: template emits leading <bos>; cell 32 strips it via
        # .removeprefix("<bos>"); the tokenizer re-adds exactly one <bos>
        # at tokenization (its default). Net result: single BOS. This is
        # the Unsloth canonical pattern and matches both Gemma 3 and
        # Gemma 4 (both emit <bos> in the template string).

        report_to = "none",
        seed = 3407,
    ),
)

print("Trainer created successfully.")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/5187 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/300 [00:00<?, ? examples/s]

Trainer created successfully.


We use Unsloth's `train_on_responses_only` so the loss is computed only on
the model's replies, not on the user/system instructions. `<|turn>user` and
`<|turn>model` are the actual Gemma 4 turn delimiters (different from Gemma 3's
`<start_of_turn>` format). After training is set up, run the verification
cells below to confirm only the assistant response is unmasked.


In [12]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part    = "<|turn>model\n",
)


Map (num_proc=64):   0%|          | 0/5187 [00:00<?, ? examples/s]

Filter (num_proc=64):   0%|          | 0/5187 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/300 [00:00<?, ? examples/s]

Filter (num_proc=64):   0%|          | 0/300 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again.  Notice how the sample only has a single `<bos>` as expected!

In [16]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<|turn>user\nআমি ট্যাম্পন ব্যবহার করার কথা ভাবছি। এগুলো কি সাধারণত নিরাপদ বলে মনে করা হয়?<turn|>\n<|turn>model\nআপু, আপনি ট্যাম্পন ব্যবহারের কথা ভাবছেন জেনে ভালো লাগলো। ট্যাম্পন সাধারণত নিরাপদ, যদি আপনি কিছু নিয়ম মেনে চলেন। অনেকেই মাসিক চলাকালীন ট্যাম্পন ব্যবহার করে স্বাচ্ছন্দ্য বোধ করেন। ট্যাম্পন ব্যবহারের ক্ষেত্রে সবচেয়ে গুরুত্বপূর্ণ বিষয় হলো সঠিক উপায়ে ব্যবহার করা এবং কিছু সতর্কতা অবলম্বন করা। ট্যাম্পন ব্যবহার করলে টক্সিক শক সিন্ড্রোম (TSS) নামক একটি বিরল কিন্তু গুরুতর অবস্থার ঝুঁকি থাকে। তবে এটি খুবই বিরল এবং সঠিক ব্যবহারবিধি মেনে চললে এর ঝুঁকি অনেক কমে যায়। **ট্যাম্পন ব্যবহারের কিছু গুরুত্বপূর্ণ দিক:** * **সঠিক শোষণ ক্ষমতা বেছে নিন:** আপনার মাসিকের প্রবাহ অনুযায়ী সঠিক শোষণ ক্ষমতার ট্যাম্পন ব্যবহার করুন। অতিরিক্ত শোষণ ক্ষমতার ট্যাম্পন দীর্ঘক্ষণ ব্যবহার করলে TSS-এর ঝুঁকি বাড়তে পারে। * **নিয়মিত পরিবর্তন করুন:** প্রতি ৪-৮ ঘণ্টা অন্তর ট্যাম্পন পরিবর্তন করা উচিত। কোনোভাবেই ৮ ঘণ্টার বেশি একটি ট্যাম্পন ব্যবহার করবেন না। রাতে ঘুমানোর সময় যদি ৮ ঘণ্টার বেশি ঘুম হয়, তাহলে প্যাড ব্যবহার করা 

Now let's print the masked out example - you should see only the answer is present:

In [17]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                           আপু, আপনি ট্যাম্পন ব্যবহারের কথা ভাবছেন জেনে ভালো লাগলো। ট্যাম্পন সাধারণত নিরাপদ, যদি আপনি কিছু নিয়ম মেনে চলেন। অনেকেই মাসিক চলাকালীন ট্যাম্পন ব্যবহার করে স্বাচ্ছন্দ্য বোধ করেন। ট্যাম্পন ব্যবহারের ক্ষেত্রে সবচেয়ে গুরুত্বপূর্ণ বিষয় হলো সঠিক উপায়ে ব্যবহার করা এবং কিছু সতর্কতা অবলম্বন করা। ট্যাম্পন ব্যবহার করলে টক্সিক শক সিন্ড্রোম (TSS) নামক একটি বিরল কিন্তু গুরুতর অবস্থার ঝুঁকি থাকে। তবে এটি খুবই বিরল এবং সঠিক ব্যবহারবিধি মেনে চললে এর ঝুঁকি অনেক কমে যায়। **ট্যাম্পন ব্যবহারের কিছু গুরুত্বপূর্ণ দিক:** * **সঠিক শোষণ ক্ষমতা বেছে নিন:** আপনার মাসিকের প্রবাহ অনুযায়ী সঠিক শোষণ ক্ষমতার ট্যাম্পন ব্যবহার করুন। অতিরিক্ত শোষণ ক্ষমতার ট্যাম্পন দীর্ঘক্ষণ ব্যবহার করলে TSS-এর ঝুঁকি বাড়তে পারে। * **নিয়মিত পরিবর্তন করুন:** প্রতি ৪-৮ ঘণ্টা অন্তর ট্যাম্পন পরিবর্তন করা উচিত। কোনোভাবেই ৮ ঘণ্টার বেশি একটি ট্যাম্পন ব্যবহার করবেন না। রাতে ঘুমানোর সময় যদি ৮ ঘণ্টার বেশি ঘুম হয়, তাহলে প্যাড ব্যবহার করা ভালো। * **পরিষ্কার-পরিচ্ছন্নতা:** ট্যাম্পন ঢোকানোর আগে এবং বের করার আগে আপনার হাত ভালো

In [18]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 5090. Max memory = 31.367 GB.
10.254 GB of memory reserved.


# Let's train the model!

To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [19]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,187 | Num Epochs = 2 | Total steps = 1,298
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 36,700,160 of 8,032,856,608 (0.46% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,3.553226,1.442780
100,1.443942,1.366215
150,1.177215,1.300732
200,1.128192,1.283521
250,1.094399,1.290746
300,1.064964,1.280918
350,1.018194,1.266306
400,0.979212,1.260284
450,0.989351,1.272979
500,0.964998,1.232101


In [20]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

6353.4765 seconds used for training.
105.89 minutes used for training.
Peak reserved memory = 13.473 GB.
Peak reserved memory for training = 3.219 GB.
Peak reserved memory % of max memory = 42.953 %.
Peak reserved memory for training % of max memory = 10.262 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-4` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [24]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "Continue the sequence: 1, 1, 2, 3, 5, 8,",
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

['<bos><|turn>user\nContinue the sequence: 1, 1, 2, 3, 5, 8,<turn|>\n<|turn>model\nThe sequence you provided is the **Fibonacci sequence**.\n\nTo continue it, you add the two previous numbers to get the next one:\n\n* $1 + 1 = 2$\n* $1 + 2 = 3$\n* $2 + 3 = 5$\n* $']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [25]:
# Menochat system prompt + a real Bangla user question
SYSTEM_PROMPT = (
    "তুমি একজন সহানুভূতিশীল, বিশ্বস্ত, এবং তথ্যনির্ভর বাংলা স্বাস্থ্য-সহকারী। "
    "সহজ, স্বাভাবিক, সম্মানজনক ও সহায়ক বাংলায় উত্তর দাও। "
    "অপ্রয়োজনীয় কঠিন শব্দ এড়িয়ে চলো। ভয় না বাড়িয়ে দরকারি সতর্কতার কথা বলো। "
    "প্রাসঙ্গিক হলে কখন ডাক্তার বা জরুরি সেবা নেওয়া দরকার তা পরিষ্কারভাবে বলবে। "
    "তথ্য বানিয়ে বলবে না। ওষুধের নাম বা ডোজ সুপারিশ করবে না — বরং ঘরোয়া যত্ন, "
    "বিশ্রাম, খাদ্যাভ্যাস, ব্যায়াম ও দৈনন্দিন অভ্যাসের পরামর্শ দাও।"
)
USER_Q = "আমার মাসিকের সময় খুব ব্যথা হয়। শ্বাস-প্রশ্বাসের ব্যায়াম কি এতে সাহায্য করতে পারে?"

messages = [{
    "role": "user",
    "content": [{"type": "text", "text": SYSTEM_PROMPT + "\n\n" + USER_Q}],
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 2048,
    temperature = 0.7, top_p = 0.9, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)


আপু, মাসিকের সময় তলপেটে ব্যথা হওয়াটা খুবই স্বাভাবিক একটা ব্যাপার, অনেক মেয়েরই এমনটা হয়। এই ব্যথাটা সত্যিই খুব কষ্টদায়ক হতে পারে। হ্যাঁ, কিছু সহজ শ্বাস-প্রশ্বাসের ব্যায়াম বা শ্বাস-প্রশ্বাসের কৌশলে উপকার পেতে পারেন। যখন পেটে ব্যথাটা খুব তীব্র হয়, তখন গভীর শ্বাস নিলে শরীরটা একটু শান্ত হয়, পেশিগুলো শিথিল হয় এবং ব্যথার ওপর একটা হালকা প্রভাব পড়তে পারে।

যখন ব্যথাটা চরমে পৌঁছাবে, তখন চেষ্টা করবেন ধীরে ধীরে নাক দিয়ে গভীর শ্বাস নিতে, যাতে পেটটা ফুলে ওঠে। আর তারপর মুখ দিয়ে ধীরে ধীরে শ্বাস ছাড়বেন, যেন পেটটা আবার নরম হয়ে যায়। এই ধরনের শ্বাস-প্রশ্বাসের ব্যায়ামগুলো মনকে শান্ত করতে আর পেশিগুলোকে শিথিল করতে সাহায্য করে, যা ব্যথার তীব্রতা কমাতে পারে। এছাড়া, কিছু ঘরোয়া উপায়েও আরাম পেতে পারেন:

*   **গরম সেঁক:** তলপেটে বা কোমরে গরম জলের বোতল, হট ওয়াটার ব্যাগ বা একটি গরম তোয়ালে দিয়ে সেঁক দিলে পেশিগুলো আলগা হয় এবং ব্যথা কমে। এটা খুব আরামদায়ক হয়।
*   **বিশ্রাম:** এই সময় শরীরকে যতটা সম্ভব বিশ্রাম দিন। ভারী কাজ বা যে কাজগুলো করতে গিয়ে ব্যথা বাড়ে, সেগুলো এড়িয়ে চলুন।
*   **হালকা ব্যায়াম:** যদি খুব বেশি ব্যথা ন

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [26]:
SAVE_DIR = "/workspace/gemma4_e4b_bn_menochat_unsloth_run1_final_v2"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved to:", SAVE_DIR)


Saved to: /workspace/gemma4_e4b_bn_menochat_unsloth_run1_final_v2


Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
model.push_to_hub("RafatK/menochat-gemma4-e4b-lora", token='')
tokenizer.push_to_hub("RafatK/menochat-gemma4-e4b-lora", token='')

README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/RafatK/menochat-gemma4-e4b-lora


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [ ]:
model.push_to_hub_merged(
    "RafatK/menochat-gemma4-e4b",
    tokenizer,
    save_method="merged_16bit",
    token=''
)

config.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /workspace/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:46<00:00, 166.67s/it]


Unsloth: Merge process complete. Saved to `/workspace/RafatK/menochat-gemma4-e4b`


In [33]:
import os, shutil

# Nuke the failed attempt
!rm -rf /tmp/unsloth_gguf_*

# Check what you have
!df -h /workspace /tmp

# Redirect /tmp writes to /workspace (symlink trick)
!rm -rf /workspace/tmp_gguf && mkdir -p /workspace/tmp_gguf
os.environ["TMPDIR"] = "/workspace/tmp_gguf"

# Also set for any subprocess llama.cpp might spawn
os.environ["TEMP"]    = "/workspace/tmp_gguf"
os.environ["TMP"]     = "/workspace/tmp_gguf"

Filesystem                    Size  Used Avail Use% Mounted on
mfs#eur-is-1.runpod.net:9421  1.1P  872T  211T  81% /workspace
overlay                        30G  5.0G   26G  17% /


In [40]:
import os
import tempfile
import shutil

# 1. Clear out anything stuck in the small /tmp folder first
# This recovers that 26GB as much as possible
for filename in os.listdir('/tmp'):
    file_path = os.path.join('/tmp', filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        pass 

# 2. Create and set the new temporary directory
new_temp = "/workspace/unsloth_temp"
os.makedirs(new_temp, exist_ok=True)

# Set environment variables for the system and llama.cpp
os.environ["TMPDIR"] = new_temp
os.environ["TEMP"] = new_temp
os.environ["TMP"] = new_temp

# Monkeypatch the tempfile module (This is the "Force" part)
tempfile.tempdir = new_temp

print(f"Temporary files will now be forced into: {tempfile.gettempdir()}")

Temporary files will now be forced into: /workspace/unsloth_temp


In [ ]:
model.push_to_hub_gguf(
    "RafatK/menochat-gemma4-e4b-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
    token = "",
    temporary_location = "/workspace/unsloth_temp",
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /workspace/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:33<00:00, 33.66s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


...

Uploading gemma-4-e4b-it.BF16-mmproj.gguf...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/RafatK/menochat-gemma4-e4b-gguf
Unsloth: Cleaning up temporary files...


'RafatK/menochat-gemma4-e4b-gguf'

In [ ]:
model.push_to_hub_gguf(
    "kingrafat82/menochat-gemma4-e4b-gguf",
    tokenizer,
    quantization_method="q4_k_m",
    token=''
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /workspace/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:13<00:00, 13.22s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:38<00:00, 38.51s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_al1ynj2t`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...


RuntimeError: Failed to convert model to GGUF: Unsloth: GGUF conversion failed: [Errno 28] No space left on device: 'gemma-4-e4b-it.BF16.gguf' -> '/tmp/unsloth_gguf_al1ynj2t_gguf/gemma-4-e4b-it.BF16.gguf'

In [27]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "gemma_4_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-4?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

I am Gemma 4, a Large Language Model developed by Google DeepMind. I am an open weights model, which means my underlying architecture and parameters are available for broader use and research.

As a large language model, I am designed to understand and generate human-like text based on the vast amount of data I was trained on. I can perform a wide variety of language tasks, such as answering questions, summarizing text, translating languages, writing different kinds of creative content, and assisting with coding.

My capabilities include processing both text and image inputs. While I generate text only, I can analyze and understand the content of images you


### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-4-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-4-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-4-finetune", tokenizer,
        token = "YOUR_HF_TOKEN"
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma_4_finetune",
        tokenizer,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "HF_ACCOUNT/gemma_4_finetune",
        tokenizer,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        token = "YOUR_HF_TOKEN",
    )

Now, use the `gemma-4-finetune.gguf` file or `gemma-4-finetune-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).